In [1]:
import sys, platform, importlib.metadata, json
print("AUDIT_KERNEL="+json.dumps({"python":sys.version,"executable":sys.executable,"platform":platform.platform(),"packages":dict(sorted((d.metadata["Name"],d.version) for d in importlib.metadata.distributions()))}))

AUDIT_KERNEL={"python": "3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]", "executable": "C:\\Users\\Miakhlov\\Documents\\GitHub\\Computational chemistry_course\\tmp\\course-restructure-20261118\\scripts\\environment\\.venv-chem\\Scripts\\python.exe", "platform": "Windows-10-10.0.19045-SP0", "packages": {"Jinja2": "3.1.6", "MarkupSafe": "3.0.4", "PyYAML": "6.0.3", "Pygments": "2.21.0", "Send2Trash": "2.2.0", "altair": "6.3.0", "anyio": "4.16.0", "argon2-cffi": "25.1.0", "argon2-cffi-bindings": "26.2.0", "arrow": "1.4.0", "asttokens": "3.0.2", "async-lru": "2.4.0", "attrs": "26.1.0", "babel": "2.18.0", "beautifulsoup4": "4.15.0", "bleach": "6.4.0", "certifi": "2026.10.23", "cffi": "2.1.1", "charset-normalizer": "3.5.2", "click": "8.5.1", "cloudpickle": "3.1.3", "colorama": "0.4.6", "comm": "0.2.3", "contourpy": "1.4.0", "cycler": "0.12.1", "debugpy": "1.8.22", "defusedxml": "0.7.1", "executing": "2.3.0", "fastjsonschema": "2.22.3", "fonttools": "4.68.0

# Разбор A. Отбор следующего кандидата

Высокое сходство с кофеином выбирает знакомый объект, но не обязательно информативный опыт. На линии A рассмотрим конфликт между близостью и полярностью; линия B перенесёт логику на формы аторвастатина без выдуманной биоактивности. Гипотеза: Парето-множество честнее единственного скрытого взвешенного score, если цели ещё обсуждаются.

Рассчитайте сходство к кофеину и TPSA для кандидатов A. Явно задайте направления критериев. Сравните каждую пару и покажите недоминируемые объекты на плоскости. Затем измените предпочтение TPSA и объясните, почему отбор изменился. Не называйте выбранный объект лекарством с лучшей эффективностью.

In [2]:
from pathlib import Path
import sys,json,math
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'programm/course.json').is_file())
sys.path.insert(0,str(ROOT/'practicuum/shared'))
from course_support import DATA,compound,esol,workspace,write_json,init_dossier
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
def publish_plot(fig,name):
    fig.savefig(OUT/name,dpi=120,bbox_inches='tight')
    display(fig)
    plt.close(fig)
OUT=workspace('M05-L02','case')


Сначала проверяем входы и единицы. Словари сохраняют названия полей вместе со значениями; список сохраняет соответствие записей. Выведите промежуточный результат перед дальнейшим анализом.

In [3]:
from chemistry import similarity,descriptors
names=['theophylline','theobromine','istradefylline']
points={n:(similarity(compound('caffeine')['smiles'],compound(n)['smiles']),-descriptors(compound(n)['smiles'])['TPSA']) for n in names}
def frontier(points):
    return [n for n,p in points.items() if not any(all(a>=b for a,b in zip(q,p)) and any(a>b for a,b in zip(q,p)) for other,q in points.items() if other!=n)]
selected=frontier(points);print(points,selected)


{'theophylline': (0.45714285714285713, -72.68), 'theobromine': (0.5294117647058824, -72.68), 'istradefylline': (0.22033898305084745, -80.28)} ['theobromine']


Теперь сравним модель с контрольным случаем. `assert` останавливает расчёт при нарушении контракта; это защита от ошибочного продолжения, а не оценка научного понимания.

In [4]:
fig,ax=plt.subplots()
for n,(s,t) in points.items():ax.scatter(s,-t);ax.annotate(n,(s,-t))
ax.set(xlabel='Tanimoto to caffeine',ylabel='TPSA / Angstrom^2');publish_plot(fig,'pareto.png')


<Figure size 640x480 with 1 Axes>

Сохраним именно полученный результат. Не переносите этот вывод на другую химическую форму, мишень или экспериментальные условия без отдельной проверки.

In [5]:
result={'criteria':'maximize similarity, minimize TPSA; teaching preference only','points':points,'frontier':selected,'kind':'calculated'}


In [6]:
write_json(OUT/'result.json',result)
from dossier_records import append_record
receipt=append_record(init_dossier(),'M05-L02','A',result,ROOT,inputs=['practicuum/shared/data/compounds.json'])
print('Versioned evidence:',receipt)


Versioned evidence: C:\Users\Miakhlov\Documents\GitHub\Computational chemistry_course\tmp\course-restructure-20261118\practicuum\evidence\20261118-full-course-windows\workspace\my_research\evidence\M05-L02\20261118T141553024005Z-A-2787fb53b5fb4a62baaf75b4e0b50000.json


Парето-анализ делает ценностный выбор явным, но не оценивает неизвестную активность. Для исследования B полезен также выбор наиболее различающихся форм, даже если они проигрывают эвристике. Далее проверим 3D-геометрию, которой нет в этих признаках.

Перед переходом ответьте: какие результаты получены вычислением, какие заданы условно, и какая независимая проверка ещё отсутствует?